In [1]:
import os
import pickle
import re

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi
from transformers import pipeline


e:\Projects\RAG\LangRAG\RAG\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import json

In [3]:
from langchain_core.documents import Document

In [ ]:

def setup_env():
    os.environ["LANGCHAIN_TRACING_V2"]= "false"
    load_dotenv(".env.RAG")
    hf_token=os.getenv("HF_TOKEN")
    return hf_token


In [5]:
def embedding():
    embd=HuggingFaceEmbeddings(model="sentence-transformers/all-mpnet-base-v2")
    return embd

In [13]:
def load_json():
    data_dir=r"E:\Projects\RAG\LangRAG\Data\Act"
    json_docs=[]
    for files in os.listdir(data_dir):
        if files.endswith(".json"):
            with open(os.path.join(data_dir, files), "r", encoding="utf-8") as file:
                data = json.load(file)
                json_docs.append(data)
    return json_docs

In [14]:

json_docs=load_json()
documents=[]
for act in json_docs:
    title = act.get("act_title","")
    year = act.get("act_year","")

    sections=act.get("sections", [])

    for sec in sections:
        content=sec.get("section_content","")
        text=f""" 
        Act: {title}
        Year: {year}
        section: {content}
        """
        documents.append(
            Document(
                page_content=text,
                metadata={
                    "act_title":title,
                    "year": year,
                    "source": act.get("source_url", "")
                }
            )
        )

In [16]:
def splitter():
 splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
  )
 docs=splitter.split_documents(documents)

 return docs

In [17]:
def create_embd():
  embedding= HuggingFaceEmbeddings(
  model_name="sentence-transformers/all-mpnet-base-v2"
  )

  return embedding